# 02 - Data Quality Analysis
Re-runs the profiling and checks from `src/preprocessing/data_validation.py` and shows the cleaning log. Nothing is deleted silently.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np
import plotly.express as px
from config.database import read_sql
pd.set_option('display.max_columns', 50)
# NOTE: results reflect whatever data the pipeline loaded (DATA_SOURCE=kaggle -> real Olist, sample -> SYNTHETIC).

In [ ]:
from src.ingestion.load_data import load_source_csvs
from src.preprocessing.data_validation import run_data_quality_checks, profile_table
frames = load_source_csvs(as_text=True)
{k: v.shape for k, v in frames.items()}

In [ ]:
checks = run_data_quality_checks(frames)
checks['status'].value_counts()

In [ ]:
checks[checks.status != 'PASS'].sort_values('failing_count', ascending=False)

In [ ]:
prof = profile_table('orders', frames['orders'])
pd.DataFrame(prof['column_profile'])

## Cleaning decisions

In [ ]:
import json
log = json.loads((ROOT / 'reports/data_quality/cleaning_log.json').read_text())
pd.DataFrame(log['entries'])

In [ ]:
read_sql("SELECT run_id, status, COUNT(*) FROM analytics.data_quality_log GROUP BY 1,2 ORDER BY 1 DESC LIMIT 10")